# When "Free" Is Not Spam: TF-IDF vs. Sentence Embeddings on Spam-Like Legitimate SMS
**NLP Project — Option 1: Spam Detection** · Section 7MA2 · Group 7 · Dr. Mohammed Talal Ahmed Mousalli

| Name | ID | Role |
|---|---|---|
| Faisal Abdullah Ali Al Shehri | 2240001737 | Team Leader |
| Saad Abdulaziz Saad Al Shehri | 2240006495 | Member 2 |
| Nawaf Rabie Hassan bin Shahbal | 2240009025 | Member 3 |

**Experimental question:** Which method better avoids false spam alerts on legitimate messages that contain spam-associated words such as "free", "win" or "call"?

**Environment:** Google Colab. Run with *Runtime → Restart and run all*.

Sections: 1 Setup · 2 Data · 3 Method 1 · 4 Method 2 · 5 Results · 6 Try it · 7 Sources

## 1. Setup

In [ ]:
# Clone the repository when running in Colab, then work from the repo root (relative paths).
import os, sys
REPO_URL = "https://github.com/FaisalAlshehri-1/Group7_NLP_Spam_Detection.git"
REPO_DIR = "Group7_NLP_Spam_Detection"

if not os.path.exists("src") and not os.path.exists(os.path.join("..", "src")):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
elif os.path.exists(os.path.join("..", "src")):
    os.chdir("..")          # opened from notebooks/ locally
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import random
import numpy as np
import pandas as pd
import sklearn, sentence_transformers

SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__,
      "| sentence-transformers", sentence_transformers.__version__)

## 2. Data
*Owner: Faisal.* UCI SMS Spam Collection (ham = 0, spam = 1).
Steps: trim whitespace → remove exact duplicates → group near-duplicate templates (lowercase, mask URLs, phone numbers, digits) → stratified group-aware split ≈ 70 / 15 / 15, so a template never crosses splits.

In [ ]:
from src.data_prep import download_dataset, load_raw, clean, add_groups, split, split_summary

raw_path = download_dataset("data/raw")
raw = load_raw(raw_path)
df, clean_report = clean(raw)
print(clean_report)

In [ ]:
df = add_groups(df)
print("Messages:", len(df), "| template groups:", df["group"].nunique())

train, dev, test = split(df)
summary = split_summary(train, dev, test)
summary

In [ ]:
# Save splits so the results can be reproduced.
os.makedirs("data/processed", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)
for name, part in [("train", train), ("dev", dev), ("test", test)]:
    part[["text", "label", "group"]].to_csv(f"data/processed/{name}.csv", index=False)
summary.to_csv("results/split_summary.csv", index=False)

### 2.1 Spam-associated words (training data only) and test subsets
Words most frequent in spam relative to ham (smoothed log-ratio of document frequency), computed on **train only**. A word must appear in at least 10 spam and 10 ham training messages and have 3+ letters: words that never occur in ham (e.g. "claim") cannot cause false alarms, so they would leave subset (a) almost empty.

In [ ]:
from src.spam_words import build_spam_word_list, ham_subsets

spam_words_df = build_spam_word_list(train, top_k=20, min_spam_docs=10, min_ham_docs=10)
SPAM_WORDS = spam_words_df["word"].tolist()
spam_words_df.to_csv("data/processed/spam_words.csv", index=False)
spam_words_df

In [ ]:
ham_with, ham_without = ham_subsets(test, SPAM_WORDS)
challenge = pd.read_csv("data/challenge_set.csv")
assert (challenge["label"] == 0).all(), "Challenge set must contain legitimate messages only"

print("(a) test ham WITH spam words:   ", len(ham_with))
print("(b) test ham WITHOUT spam words:", len(ham_without))
print("(c) challenge set:              ", len(challenge))

## 3. Method 1 — TF-IDF + Logistic Regression
*Owner: Saad.* Word TF-IDF (sublinear tf) → class-weighted Logistic Regression.
Grid: `C` ∈ {0.1 … 100} × n-grams ∈ {(1,1), (1,2)}. Every model is fitted on **train**; the best setting is chosen by **dev Macro F1**. Test data is not used here.

In [ ]:
from src.models import tune_tfidf, tfidf_top_words

tfidf_model, tfidf_grid = tune_tfidf(train, dev)
tfidf_grid.to_csv("results/tfidf_dev_grid.csv", index=False)
best = tfidf_grid.iloc[0]
print(f"Selected: C = {best['C']}, ngram_range = {best['ngram_range']}, dev Macro F1 = {best['dev_macro_f1']}")
tfidf_grid.head(5)

## 4. Method 2 — Sentence embeddings + Logistic Regression
*Owner: Nawaf.* Pretrained `all-MiniLM-L6-v2` (frozen, 384-d, L2-normalised) → class-weighted Logistic Regression. `C` chosen by **dev Macro F1**. The encoder weights are downloaded from Hugging Face; no fine-tuning.

In [ ]:
from src.models import load_encoder, embed, tune_embedding, EmbeddingSpamClassifier, EMB_MODEL_NAME

encoder = load_encoder(EMB_MODEL_NAME)
X_train = embed(encoder, train["text"])
X_dev = embed(encoder, dev["text"])
print("Embedding shape:", X_train.shape)

emb_clf, emb_grid = tune_embedding(X_train, train["label"], X_dev, dev["label"])
emb_grid.to_csv("results/embedding_dev_grid.csv", index=False)
print(f"Selected: C = {emb_grid.iloc[0]['C']}, dev Macro F1 = {emb_grid.iloc[0]['dev_macro_f1']}")
emb_model = EmbeddingSpamClassifier(encoder, emb_clf)
emb_grid.head(5)

In [ ]:
# Save trained models so "Try it" works without retraining.
import joblib
os.makedirs("models", exist_ok=True)
joblib.dump(tfidf_model, "models/tfidf_lr.joblib")
joblib.dump(emb_clf, "models/embed_lr.joblib")   # encoder is re-loaded by name
print("Saved models/tfidf_lr.joblib and models/embed_lr.joblib")

## 5. Results
*Owner: Saad (evaluation of both methods).* Both methods on **identical** examples.
Main comparison: Accuracy and Macro F1 on the unchanged test set. Then the false-positive rate (legitimate messages flagged as spam) on (a) test ham with spam-associated words, (b) test ham without them, (c) the hand-written challenge set. (a) vs (b) isolates the effect of the words.

In [ ]:
from src.analysis import evaluate_all, errors, predictions_side_by_side
from src.evaluation import confusion

models = {"TF-IDF + LR": tfidf_model, "MiniLM + LR": emb_model}
if len(challenge) == 0:
    print("WARNING: data/challenge_set.csv is empty; FPR (c) will be NaN until the team adds its messages.")

results_df = evaluate_all(models, test, ham_with, ham_without, challenge)
results_df.to_csv("results/main_results.csv")
results_df.round(4)

In [ ]:
for name, model in models.items():
    print(name); display(confusion(test["label"], model.predict(test["text"]))); print()

In [ ]:
import matplotlib.pyplot as plt

fpr_cols = [c for c in results_df.columns if c.startswith("FPR")]
ax = results_df.loc[list(models), fpr_cols].T.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_ylabel("False-positive rate"); ax.set_title("Legitimate messages flagged as spam")
ax.set_xticklabels(["(a) test ham\nwith words", "(b) test ham\nwithout words", "(c) challenge\nset"])
plt.tight_layout(); plt.savefig("results/fpr_chart.png", dpi=150); plt.show()

### 5.1 Error analysis
Wrong predictions on the test set for each method. For TF-IDF, `top_words` lists the words that pushed the message towards spam (coefficient × TF-IDF value). Each member explains at least three errors of their own method in the report.

In [ ]:
tfidf_err = errors(tfidf_model, test)
tfidf_err["top_words"] = tfidf_err["text"].map(lambda t: tfidf_top_words(tfidf_model, t))
tfidf_err.to_csv("results/errors_tfidf.csv", index=False)
print("TF-IDF errors on test:", len(tfidf_err))
tfidf_err.head(10)

In [ ]:
emb_err = errors(emb_model, test)
emb_err.to_csv("results/errors_embedding.csv", index=False)
print("Embedding errors on test:", len(emb_err))
emb_err.head(10)

In [ ]:
# False alarms on legitimate messages that contain spam-associated words: where do the methods disagree?
pool = pd.concat([ham_with[["text", "label"]].assign(source="test ham (a)"),
                  challenge[["text", "label"]].assign(source="challenge (c)")], ignore_index=True)
side = predictions_side_by_side(models, pool[["text", "label"]]).assign(source=pool["source"])
disagree = side[side["TF-IDF + LR"] != side["MiniLM + LR"]]
disagree.to_csv("results/disagreements.csv", index=False)
print("Disagreements:", len(disagree))
disagree.head(10)

### 5.2 Original labelled examples (shown separately from test scores)

In [ ]:
originals = pd.read_csv("data/original_examples.csv")
if len(originals):
    display(predictions_side_by_side(models, originals).assign(author=originals["author"]))
else:
    print("data/original_examples.csv is empty; each member adds two labelled examples.")

## 6. Try it
*Owner: Nawaf.* Loads the **saved** models (no retraining) and classifies a new message with both methods. Empty input returns a helpful message.

In [ ]:
from src.predict import load_models, classify

saved_models = load_models()

message = "Are you free tomorrow? Call me after class"  # @param {type:"string"}
print(classify(message, saved_models))
print()
print(classify("   ", saved_models))

## 7. Sources
- Dataset: T. A. Almeida, J. M. G. Hidalgo, A. Yamakami, ACM DocEng 2011 — https://archive.ics.uci.edu/dataset/228/sms+spam+collection
- N. Reimers, I. Gurevych, Sentence-BERT, EMNLP-IJCNLP 2019 — pretrained weights `sentence-transformers/all-MiniLM-L6-v2`
- M. T. Ribeiro et al., CheckList, ACL 2020 — motivation for the challenge set
- Libraries: scikit-learn, pandas, numpy, sentence-transformers, joblib
- AI assistance: Claude (Anthropic) helped draft the code structure and helper functions; the team reviewed, ran and is responsible for all code and results.